# Member 4: CTC, hybrid CTC/attention and RNN-T baselines (ESPnet, Colab)

Runtime: **Runtime → Change runtime type → T4 GPU**. Run the cells from the top.
Checkpoints go to Google Drive, so a disconnect does not lose training: re-run the setup cells, then the training cell with `--stage 11` and it resumes from the last epoch.

All scripts are in `baselines/ctc_attention_rnnt/` (see its README). Scores are **not** computed here: Member 2 runs `src/evaluate.py` on the prediction CSVs.

## 1. Setup (every session, about 10 minutes)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Branch with the Member 4 code (change if yours has another name).
BRANCH = 'amro-ctc-attention-rnnt'
EXP = '/content/drive/MyDrive/urdu_asr_member4/exp'   # checkpoints, kept across sessions
import os
os.environ['BRANCH'] = BRANCH
os.environ['EXP'] = EXP
os.environ['ESPNET_ROOT'] = '/content/espnet'

In [ ]:
%%bash
cd /content
[ -d urdu-asr-kathbath ] || git clone https://github.com/r4fo/urdu-asr-kathbath.git
cd urdu-asr-kathbath
git fetch origin && git checkout "$BRANCH" && git pull
bash baselines/ctc_attention_rnnt/setup_espnet.sh "$ESPNET_ROOT"

## 2. Pipeline test on 50 clips (a few minutes)

40 train, 10 dev and 10 test clips from `data/clean/`, speaker-disjoint, 2 epochs. The output text is garbage after 2 epochs; this only checks that every stage runs. Nothing from this test is committed or reported.

In [ ]:
%%bash
cd /content/urdu-asr-kathbath
python3 baselines/ctc_attention_rnnt/make_sample_splits.py
for s in ctc hybrid rnnt; do
  bash baselines/ctc_attention_rnnt/run.sh $s --name sample \
      --splits-dir baselines/ctc_attention_rnnt/work/sample_splits \
      --asr-args "--max_epoch 2" > /content/sample_$s.log 2>&1 \
    && echo "$s OK" || { echo "$s FAILED, last lines:"; tail -30 /content/sample_$s.log; }
done
head -3 baselines/ctc_attention_rnnt/work/sample/predictions_dev/*.csv

## 3. Train one system on the real split (a few hours each)

Needs `data/splits/train.csv`, `dev.csv` (Member 2, frozen Thursday). Pick `SYSTEM` = `ctc`, `hybrid` or `rnnt` and run. The best epoch is the one with the lowest dev loss (`valid.loss.best.pth`).

After a disconnect: run section 1 again, then this cell with `STAGE=11`. It rebuilds the local audio dump and resumes training from the checkpoint on Drive.

In [ ]:
%%bash
SYSTEM=ctc     # ctc | hybrid | rnnt
STAGE=2        # 2 = from the start, 11 = resume training, 12 = decode only
cd /content/urdu-asr-kathbath
bash baselines/ctc_attention_rnnt/run.sh $SYSTEM --expdir "$EXP" --stage $STAGE 2>&1 | tail -40

Dev predictions (for your own checks and error analysis) and timing notes, copied to Drive:

In [ ]:
%%bash
cd /content/urdu-asr-kathbath/baselines/ctc_attention_rnnt/work/full
mkdir -p "$EXP/../outputs" && cp -r predictions_dev timing_*.txt "$EXP/../outputs/" 2>/dev/null; ls "$EXP/../outputs"
tail -n 5 timing_*.txt

## 4. Test, ONCE, after all settings are frozen on dev

Writes `results/predictions/<system>.csv`. Copy the three CSVs and the timing notes to your computer and commit them from your own account (or download them from Drive).

In [ ]:
%%bash
SYSTEM=ctc     # run once for each of ctc, hybrid, rnnt
cd /content/urdu-asr-kathbath
bash baselines/ctc_attention_rnnt/run.sh $SYSTEM --expdir "$EXP" --stage 12 --test 2>&1 | tail -20
mkdir -p "$EXP/../outputs/predictions_test"
cp results/predictions/$SYSTEM.csv "$EXP/../outputs/predictions_test/"
cp baselines/ctc_attention_rnnt/work/full/timing_$SYSTEM.txt "$EXP/../outputs/"